# 02. Limpieza de datos

Este notebook contiene la limpieza del dataset de víctimas del conflicto armado en Colombia.

## Importación de Librerías

Importamos las librerías necesarias para el análisis de datos.

In [ ]:
import pandas as pd
import csv
import os
import re
import matplotlib.pyplot as plt
import seaborn as sns
import calendar

## Carga del Dataset

Cargamos el dataset de víctimas del conflicto armado desde el archivo CSV.

In [2]:
# Define the relative path to the data file
data_path = '../data/raw/victimas_por_hechos_departamental_20250416.csv'

# Load the dataset
try:
    # Attempt to read with UTF-8 encoding first
    df = pd.read_csv(data_path, encoding='utf-8') 
except UnicodeDecodeError:
    try:
        # Fallback to Latin-1 encoding if UTF-8 fails
        df = pd.read_csv(data_path, encoding='latin1') 
    except Exception as e:
        print(f"Error loading CSV file: {e}")
        df = None # Indicate failure by setting df to None

# Verificar si el dataset se cargó correctamente
if df is not None:
    print(f"Dataset cargado exitosamente. Dimensiones: {df.shape}")
else:
    print("Error al cargar el dataset. Por favor, verifica la ruta y la codificación del archivo.")

Dataset cargado exitosamente. Dimensiones: (2431164, 17)


# Eliminacion de columnas innecesarias
Se eliminaran las columnas que no sean necesarias para el analisis como NOM_RPT, COD_PAI y PAIS

In [3]:
# Mostrar forma inicial del dataframe y vista previa
print(f"Forma inicial del dataframe: {df.shape}")
print(f"Columnas del dataframe: {df.columns.tolist()}")

Forma inicial del dataframe: (2431164, 17)
Columnas del dataframe: ['FECHA_CORTE', 'NOM_RPT', 'COD_PAIS', 'PAIS', 'COD_ESTADO_DEPTO', 'ESTADO_DEPTO', 'PARAM_HECHO', 'HECHO', 'SEXO', 'ETNIA', 'DISCAPACIDAD', 'CICLO_VITAL', 'PER_OCU', 'PER_DECLA', 'PER_UBIC', 'PER_SA', 'EVENTOS']


In [4]:
# Columnas que se eliminarán
columns_to_drop = ["NOM_RPT", "COD_PAIS", "PAIS"]

# Eliminar columnas especificadas
df = df.drop(columns=columns_to_drop)

In [6]:
# Mostrar forma final del dataframe
print(f"Forma final del dataframe después de eliminar columnas: {df.shape}")
print(f"Columnas del dataframe: {df.columns.tolist()}")
df.head()

Forma final del dataframe después de eliminar columnas: (2431164, 14)
Columnas del dataframe: ['FECHA_CORTE', 'COD_ESTADO_DEPTO', 'ESTADO_DEPTO', 'PARAM_HECHO', 'HECHO', 'SEXO', 'ETNIA', 'DISCAPACIDAD', 'CICLO_VITAL', 'PER_OCU', 'PER_DECLA', 'PER_UBIC', 'PER_SA', 'EVENTOS']


,FECHA_CORTE,COD_ESTADO_DEPTO,ESTADO_DEPTO,PARAM_HECHO,HECHO,SEXO,ETNIA,DISCAPACIDAD,CICLO_VITAL,PER_OCU,PER_DECLA,PER_UBIC,PER_SA,EVENTOS
0,2022/03/31 00:00:00.000000000,13,Bolivar,5,Desplazamiento forzado,Hombre,Gitano (RROM) (Acreditado RA),Ninguna,entre 18 y 28,3.0,4.0,3.0,3.0,3.0
1,2022/02/28 00:00:00.000000000,20,Cesar,5,Desplazamiento forzado,Mujer,Raizal del Archipielago de San Andres y Provid...,Multiple,entre 61 y 100,1.0,NaN,NaN,NaN,1.0
2,2022/04/30 00:00:00.000000000,18,Caqueta,5,Desplazamiento forzado,Hombre,Gitano (RROM) (Acreditado RA),Ninguna,entre 29 y 60,2.0,NaN,NaN,NaN,2.0
3,2022/05/31 00:00:00.000000000,95,Guaviare,5,Desplazamiento forzado,Hombre,Negro(a) o Afrocolombiano(a),Fisica,entre 61 y 100,11.0,7.0,4.0,4.0,12.0
4,2022/04/30 00:00:00.000000000,18,Caqueta,2,Amenaza,Hombre,Indigena (Acreditado RA),Por Establecer,entre 12 y 17,1.0,1.0,1.0,1.0,1.0


# Detectar patrones en formato de FECHA_CORTE
Acá reemplazamos los digitos de la fecha por "d", asi los patrones se verian mejor

In [7]:
# Supongamos que tu DataFrame se llama df
df["FORMATO_FECHA"] = df["FECHA_CORTE"].astype(str).str.replace(r"\d", "d", regex=True)

# Ver los primeros formatos únicos
print(df["FORMATO_FECHA"].value_counts().head(10))

FORMATO_FECHA
dddd/dd/dd dd:dd:dd.ddddddddd    2048672
dd/dd/dddd                        319076
dd/dd/dd                           63416
Name: count, dtype: int64


In [8]:
df[df["FORMATO_FECHA"] == 'dddd/dd/dd dd:dd:dd.ddddddddd']

,FECHA_CORTE,COD_ESTADO_DEPTO,ESTADO_DEPTO,PARAM_HECHO,HECHO,SEXO,ETNIA,DISCAPACIDAD,CICLO_VITAL,PER_OCU,PER_DECLA,PER_UBIC,PER_SA,EVENTOS,FORMATO_FECHA
0,2022/03/31 00:00:00.000000000,13,Bolivar,5,Desplazamiento forzado,Hombre,Gitano (RROM) (Acreditado RA),Ninguna,entre 18 y 28,3.0,4.0,3.0,3.0,3.0,dddd/dd/dd dd:dd:dd.ddddddddd
1,2022/02/28 00:00:00.000000000,20,Cesar,5,Desplazamiento forzado,Mujer,Raizal del Archipielago de San Andres y Provid...,Multiple,entre 61 y 100,1.0,NaN,NaN,NaN,1.0,dddd/dd/dd dd:dd:dd.ddddddddd
2,2022/04/30 00:00:00.000000000,18,Caqueta,5,Desplazamiento forzado,Hombre,Gitano (RROM) (Acreditado RA),Ninguna,entre 29 y 60,2.0,NaN,NaN,NaN,2.0,dddd/dd/dd dd:dd:dd.ddddddddd
3,2022/05/31 00:00:00.000000000,95,Guaviare,5,Desplazamiento forzado,Hombre,Negro(a) o Afrocolombiano(a),Fisica,entre 61 y 100,11.0,7.0,4.0,4.0,12.0,dddd/dd/dd dd:dd:dd.ddddddddd
4,2022/04/30 00:00:00.000000000,18,Caqueta,2,Amenaza,Hombre,Indigena (Acreditado RA),Por Establecer,entre 12 y 17,1.0,1.0,1.0,1.0,1.0,dddd/dd/dd dd:dd:dd.ddddddddd
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2060050,2024/08/31 00:00:00.000000000,17,Caldas,2,Amenaza,Hombre,Negro(a) o Afrocolombiano(a),Por Establecer,entre 60 y 110,1.0,NaN,NaN,NaN,1.0,dddd/dd/dd dd:dd:dd.ddddddddd
2060051,2024/08/31 00:00:00.000000000,76,Valle del Cauca,2,Amenaza,Mujer,Negro(a) o Afrocolombiano(a),Fisica,entre 6 y 11,5.0,14.0,12.0,12.0,5.0,dddd/dd/dd dd:dd:dd.ddddddddd
2060052,2024/08/31 00:00:00.000000000,66,Risaralda,2,Amenaza,Hombre,Indigena,Visual,entre 29 y 59,NaN,1.0,NaN,NaN,NaN,dddd/dd/dd dd:dd:dd.ddddddddd
2060053,2024/08/31 00:00:00.000000000,66,Risaralda,5,Desplazamiento forzado,Hombre,Gitano(a) ROM,Ninguna,entre 18 y 28,1.0,7.0,7.0,6.0,1.0,dddd/dd/dd dd:dd:dd.ddddddddd


In [9]:
df[df["FORMATO_FECHA"] == 'dd/dd/dddd']

,FECHA_CORTE,COD_ESTADO_DEPTO,ESTADO_DEPTO,PARAM_HECHO,HECHO,SEXO,ETNIA,DISCAPACIDAD,CICLO_VITAL,PER_OCU,PER_DECLA,PER_UBIC,PER_SA,EVENTOS,FORMATO_FECHA
100,31/01/2025,0,SIN DEFINIR,2,Amenaza,Hombre,Negro(a) o Afrocolombiano(a),Fisica,entre 6 y 11,NaN,NaN,2.0,NaN,NaN,dd/dd/dddd
342,31/01/2025,13,Bolivar,6,Homicidio,Mujer,Ninguna,Ninguna,ND,313.0,236.0,47.0,38.0,315.0,dd/dd/dddd
523,31/01/2025,13,Bolivar,6,Homicidio,Mujer,Ninguna,Ninguna,entre 60 y 110,3304.0,2323.0,2607.0,2226.0,3528.0,dd/dd/dddd
1044,31/01/2025,13,Bolivar,6,Homicidio,Mujer,Ninguna,Ninguna,entre 29 y 59,6050.0,4218.0,4575.0,4387.0,6269.0,dd/dd/dddd
1185,31/01/2025,13,Bolivar,6,Homicidio,Mujer,Ninguna,Por Establecer,entre 18 y 28,5.0,2.0,3.0,3.0,5.0,dd/dd/dddd
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2431159,28/02/2025,18,Caqueta,6,Homicidio,Mujer,Indigena,Ninguna,entre 18 y 28,17.0,12.0,13.0,12.0,18.0,dd/dd/dddd
2431160,28/02/2025,23,Cordoba,13,Lesiones Personales Fisicas,Mujer,Negro(a) o Afrocolombiano(a),Ninguna,entre 18 y 28,1.0,1.0,1.0,1.0,1.0,dd/dd/dddd
2431161,28/02/2025,5,Antioquia,1,Acto terrorista / Atentados / Combates / Enfre...,Hombre,Ninguna,Intelectual,entre 60 y 110,4.0,5.0,5.0,4.0,4.0,dd/dd/dddd
2431162,28/02/2025,66,Risaralda,12,Perdida de Bienes Muebles o Inmuebles,Hombre,Indigena (Acreditado RA),Por Establecer,entre 60 y 110,1.0,1.0,1.0,NaN,1.0,dd/dd/dddd


In [10]:
df[df["FORMATO_FECHA"] == 'dd/dd/dd']

,FECHA_CORTE,COD_ESTADO_DEPTO,ESTADO_DEPTO,PARAM_HECHO,HECHO,SEXO,ETNIA,DISCAPACIDAD,CICLO_VITAL,PER_OCU,PER_DECLA,PER_UBIC,PER_SA,EVENTOS,FORMATO_FECHA
20,30/09/24,0,SIN DEFINIR,2,Amenaza,Hombre,Indigena (Acreditado RA),Multiple,entre 18 y 28,NaN,NaN,3.0,3.0,NaN,dd/dd/dd
444,30/09/24,0,SIN DEFINIR,2,Amenaza,Hombre,Indigena (Acreditado RA),Multiple,entre 60 y 110,NaN,3.0,4.0,2.0,NaN,dd/dd/dd
807,30/09/24,50,Meta,6,Homicidio,Mujer,Ninguna,Ninguna,entre 29 y 59,8996.0,9169.0,8252.0,7917.0,9508.0,dd/dd/dd
967,30/09/24,50,Meta,6,Homicidio,Mujer,Ninguna,Ninguna,entre 60 y 110,4414.0,4563.0,4050.0,3427.0,4783.0,dd/dd/dd
1090,30/09/24,0,SIN DEFINIR,2,Amenaza,Hombre,Indigena (Acreditado RA),Multiple,entre 29 y 59,NaN,2.0,8.0,8.0,NaN,dd/dd/dd
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2120146,30/09/24,99,Vichada,15,Confinamiento,Hombre,Negro(a) o Afrocolombiano(a),Ninguna,entre 12 y 17,NaN,NaN,1.0,1.0,NaN,dd/dd/dd
2120147,30/09/24,99,Vichada,15,Confinamiento,Hombre,Ninguna,Ninguna,entre 29 y 59,NaN,NaN,1.0,1.0,NaN,dd/dd/dd
2120148,30/09/24,99,Vichada,15,Confinamiento,Mujer,Indigena,Ninguna,entre 29 y 59,NaN,NaN,1.0,1.0,NaN,dd/dd/dd
2120149,30/09/24,99,Vichada,15,Confinamiento,Mujer,Ninguna,Ninguna,entre 18 y 28,NaN,NaN,2.0,2.0,NaN,dd/dd/dd


## Transformacion y Normalizacion de las fechas

### Se elimina datos innecesarios como hora, minutos y segundos

In [11]:
df["FECHA_CORTE"] = df["FECHA_CORTE"].str.split(" ").str[0]

### Aseguramos que todo sea string

In [12]:
df["FECHA_CORTE"] = df["FECHA_CORTE"].astype(str).str.strip()

### Función para normalizar fechas a texto

In [13]:
def normalizar_fecha(fecha_str: str) -> str:
    """
    Normaliza fechas a formato dd/mm/yyyy (texto).
    Soporta variaciones: dd/mm/yy, dd/mm/yyyy, yyyy/mm/dd, yyyy-mm-dd.
    """
    if not isinstance(fecha_str, str):
        return fecha_str
    
    # Separar en partes por /, -, . o espacio
    partes = re.split(r"[-/.\s]", fecha_str.strip())
    partes = [p for p in partes if p]  # eliminar vacíos
    
    if len(partes) < 3:
        return fecha_str  # no parece fecha válida
    
    # Caso: yyyy/mm/dd
    if len(partes[0]) == 4:
        anio, mes, dia = partes[0], partes[1], partes[2]
    else:
        # Caso: dd/mm/yyyy o dd/mm/yy
        dia, mes, anio = partes[0], partes[1], partes[2]
    
    # Rellenar con ceros
    dia = dia.zfill(2)
    mes = mes.zfill(2)
    
    # Año: expandir si viene en 2 dígitos
    if len(anio) == 2:
        # puedes ajustar la regla: aquí asumo 2000+
        anio = "20" + anio
    
    return f"{dia}/{mes}/{anio}"

# # Probar
# ejemplos = ["30/09/24", "28/02/2025", "2024/08/31"]
# for e in ejemplos:
#     print(e, "->", normalizar_fecha(e))


In [14]:
df["FECHA_CORTE"] = df["FECHA_CORTE"].apply(normalizar_fecha)

In [15]:
df = df.drop(columns=["FORMATO_FECHA"])

In [16]:
df['FECHA_CORTE'] = pd.to_datetime(df['FECHA_CORTE'], format="%d/%m/%Y")

In [ ]:
# Asegúrate que FECHA_CORTE es datetime
df["FECHA_CORTE"] = pd.to_datetime(df["FECHA_CORTE"])

# Recorremos por año y mes únicos en orden
for (anio, mes), subdf in df.groupby([df["FECHA_CORTE"].dt.year,
                                      df["FECHA_CORTE"].dt.month]):
    nombre_mes = calendar.month_name[mes]  # Para mostrar "January", "February", etc.
    print(f"\n===== {nombre_mes} - {anio} =====")
    print(subdf["FECHA_CORTE"].value_counts())



===== January - 2022 =====
FECHA_CORTE
2022-01-31    113320
Name: count, dtype: int64

===== February - 2022 =====
FECHA_CORTE
2022-02-28    56804
Name: count, dtype: int64

===== March - 2022 =====
FECHA_CORTE
2022-03-31    56783
Name: count, dtype: int64

===== April - 2022 =====
FECHA_CORTE
2022-04-30    56845
Name: count, dtype: int64

===== May - 2022 =====
FECHA_CORTE
2022-05-31    56964
Name: count, dtype: int64

===== June - 2022 =====
FECHA_CORTE
2022-06-30    114172
Name: count, dtype: int64

===== July - 2022 =====
FECHA_CORTE
2022-07-31    57075
Name: count, dtype: int64

===== August - 2022 =====
FECHA_CORTE
2022-08-31    57221
Name: count, dtype: int64

===== September - 2022 =====
FECHA_CORTE
2022-09-30    57150
Name: count, dtype: int64

===== October - 2022 =====
FECHA_CORTE
2022-10-31    57270
Name: count, dtype: int64

===== November - 2022 =====
FECHA_CORTE
2022-11-30    57423
Name: count, dtype: int64

===== December - 2022 =====
FECHA_CORTE
2022-12-31    57503
Na

# Corrección de errores de codificación y unificación de categorías

Se identificaron y reemplazaron valores con errores de codificación en varias columnas, con el fin de mejorar la calidad y consistencia de los datos. A continuación, se detallan los ajustes realizados:

### Columna: `ESTADO_DEPTO`

Valores corregidos:

* `"Nari�o"`
* `"NariÃ±o"`

Reemplazados por:

* `"Nariño"`

---

### Columna: `HECHO`

Valores corregidos:

* `"Desaparici�n forzada"`
* `"DesapariciÃ³n forzada"`

Reemplazados por:

* `"Desaparición forzada"`

---

* `"VinculaciÃ³n de NiÃ±os NiÃ±as y Adolescentes a Actividades Relacionadas con grupos armados"`
* `"Vinculaci�n de Ni�os Ni�as y Adolescentes a Actividades Relacionadas con grupos armados"`

Reemplazados por:

* `"Vinculación de Niños Niñas y Adolescentes a Actividades Relacionadas con grupos armados"`

---

* `"Minas Antipersonal, Munici�n sin Explotar y Artefacto Explosivo improvisado"`

Reemplazado por:

* `"Minas Antipersonal, Munición sin Explotar y Artefacto Explosivo improvisado"`

---

### Columna: `ETNIA`

Unificación de categorías para facilitar el análisis:

Valores originales:

* `"Negro (Acreditado RA)"`
* `"Afrocolombiano (Acreditado RA)"`

Reemplazados por:

* `"Negro(a) o Afrocolombiano(a) (Acreditado RA)"`


In [18]:
# Corrección de errores de codificación

df['ESTADO_DEPTO'] = df['ESTADO_DEPTO'].replace({
    'Nari�o': 'Nariño',
    'NariÃ±o': 'Nariño'
})

df['HECHO'] = df['HECHO'].replace({
    'Desaparici�n forzada': 'Desaparición forzada',
    'DesapariciÃ³n forzada': 'Desaparición forzada',
    'VinculaciÃ³n de NiÃ±os NiÃ±as y Adolescentes a Actividades Relacionadas con grupos armados':
        'Vinculación de Niños Niñas y Adolescentes a Actividades Relacionadas con grupos armados',
    'Vinculaci�n de Ni�os Ni�as y Adolescentes a Actividades Relacionadas con grupos armados':
        'Vinculación de Niños Niñas y Adolescentes a Actividades Relacionadas con grupos armados',
    'Minas Antipersonal, Munici�n sin Explotar y Artefacto Explosivo improvisado':
        'Minas Antipersonal, Munición sin Explotar y Artefacto Explosivo improvisado'
})

df['ETNIA'] = df['ETNIA'].replace({
    'Negro (Acreditado RA)': 'Negro(a) o Afrocolombiano(a) (Acreditado RA)',
    'Afrocolombiano (Acreditado RA)': 'Negro(a) o Afrocolombiano(a) (Acreditado RA)'
})

df['CICLO_VITAL'] = df['CICLO_VITAL'].replace({
'entre 61 y 100': 'entre 60 y 110'
})


# Limpieza y Validación de Grupos Únicos para Reemplazo de NaN por Cero

Este análisis tiene como objetivo **justificar y ejecutar** el reemplazo de valores faltantes (`NaN`) por `0` en las variables de conteo:

- `PER_OCU`
- `PER_DECLA`
- `PER_UBIC`
- `PER_SA`
- `EVENTOS`

El procedimiento se realiza **únicamente si se confirma que no existen registros duplicados** por grupo definido a partir de:

- `FECHA_CORTE`
- `HECHO`
- `ESTADO_DEPTO`
- `SEXO`
- `CICLO_VITAL`
- `ETNIA`

De esta forma se garantiza que cada fila representa un grupo único y que los valores nulos pueden interpretarse como **"cero personas/eventos registrados"**.



In [19]:
# Paso 1: Definir columnas
columnas_grupo = ['COD_ESTADO_DEPTO','ESTADO_DEPTO', 'PARAM_HECHO','HECHO', 'SEXO',  'ETNIA', 'DISCAPACIDAD', 'CICLO_VITAL']
variables_conteo = ['PER_OCU', 'PER_DECLA', 'PER_UBIC', 'PER_SA', 'EVENTOS']

## Verificamos si existe el valor 0 en los datos
Con esto nos aseguramos que los valores NaN son la ausencia de reportes y no perdida de informacion

In [20]:
for column in variables_conteo:
    cero_count = len(df[df[column] == 0])
    if cero_count > 0:
        print(f'La columna {column} tiene {cero_count} valores 0')
    else:
        print(f'no hay valores 0 en {column}')

no hay valores 0 en PER_OCU
no hay valores 0 en PER_DECLA
no hay valores 0 en PER_UBIC
no hay valores 0 en PER_SA
no hay valores 0 en EVENTOS


In [21]:
# Paso 3: Reemplazar NaN por 0
df[variables_conteo] = df[variables_conteo].fillna(0)

In [22]:
# cambio de tipo float a int en variables de conteo
df[variables_conteo] = df[variables_conteo].astype(int)

df.dtypes

FECHA_CORTE         datetime64[ns]
COD_ESTADO_DEPTO             int64
ESTADO_DEPTO                object
PARAM_HECHO                  int64
HECHO                       object
SEXO                        object
ETNIA                       object
DISCAPACIDAD                object
CICLO_VITAL                 object
PER_OCU                      int64
PER_DECLA                    int64
PER_UBIC                     int64
PER_SA                       int64
EVENTOS                      int64
dtype: object

# Generacion de COD_RPT de victimas
En esta seccion se genera el COD_RPT de las victimas para su posterior uso en la generacion de las primary keys.

In [26]:
# Asegurarse de que el índice sea limpio, secuencial y comience desde 0
df = df.reset_index(drop=True)

# Crear columna 'COD_RPT' como ID incremental comenzando desde 1
df['COD_RPT'] = df.index + 1

# Reordenar columnas para que 'COD_RPT' esté al inicio
df = df[['COD_RPT'] + [col for col in df.columns if col != 'COD_RPT']]


# Muestra del dataset de víctimas limpio 

In [29]:
df.head()

,COD_RPT,FECHA_CORTE,COD_ESTADO_DEPTO,ESTADO_DEPTO,PARAM_HECHO,HECHO,SEXO,ETNIA,DISCAPACIDAD,CICLO_VITAL,PER_OCU,PER_DECLA,PER_UBIC,PER_SA,EVENTOS
0,1,2022-03-31,13,Bolivar,5,Desplazamiento forzado,Hombre,Gitano (RROM) (Acreditado RA),Ninguna,entre 18 y 28,3,4,3,3,3
1,2,2022-02-28,20,Cesar,5,Desplazamiento forzado,Mujer,Raizal del Archipielago de San Andres y Provid...,Multiple,entre 60 y 110,1,0,0,0,1
2,3,2022-04-30,18,Caqueta,5,Desplazamiento forzado,Hombre,Gitano (RROM) (Acreditado RA),Ninguna,entre 29 y 60,2,0,0,0,2
3,4,2022-05-31,95,Guaviare,5,Desplazamiento forzado,Hombre,Negro(a) o Afrocolombiano(a),Fisica,entre 60 y 110,11,7,4,4,12
4,5,2022-04-30,18,Caqueta,2,Amenaza,Hombre,Indigena (Acreditado RA),Por Establecer,entre 12 y 17,1,1,1,1,1


In [30]:
df.tail()

,COD_RPT,FECHA_CORTE,COD_ESTADO_DEPTO,ESTADO_DEPTO,PARAM_HECHO,HECHO,SEXO,ETNIA,DISCAPACIDAD,CICLO_VITAL,PER_OCU,PER_DECLA,PER_UBIC,PER_SA,EVENTOS
2431159,2431160,2025-02-28,18,Caqueta,6,Homicidio,Mujer,Indigena,Ninguna,entre 18 y 28,17,12,13,12,18
2431160,2431161,2025-02-28,23,Cordoba,13,Lesiones Personales Fisicas,Mujer,Negro(a) o Afrocolombiano(a),Ninguna,entre 18 y 28,1,1,1,1,1
2431161,2431162,2025-02-28,5,Antioquia,1,Acto terrorista / Atentados / Combates / Enfre...,Hombre,Ninguna,Intelectual,entre 60 y 110,4,5,5,4,4
2431162,2431163,2025-02-28,66,Risaralda,12,Perdida de Bienes Muebles o Inmuebles,Hombre,Indigena (Acreditado RA),Por Establecer,entre 60 y 110,1,1,1,0,1
2431163,2431164,2025-02-28,5,Antioquia,15,Confinamiento,Hombre,Indigena (Acreditado RA),Por Establecer,entre 29 y 59,2,2,2,2,3


# Exportamos los datos limpios a un archivo CSV
Se exporta el archivo CSV con los datos limpios en en ´/data/processed/victimas_por_hechos_departamental_20250416.csv´ para su posterior análisis

In [112]:
df.to_csv(
    path_or_buf='../data/processed/victimas_por_hechos_departamental_20250416.csv',
    sep=',',
    na_rep='',
    header=True,
    index=False,
    encoding='utf-8',
    quoting=csv.QUOTE_MINIMAL,
    lineterminator=os.linesep,
    quotechar='"',
    decimal='.',
    errors='strict'
)